# UFPR-ALPR: Three-model evaluation on 100 test images

Compares YOLOv8, Faster R-CNN, and SSD300 on the same test images, using the same fine-tuned PARSeq OCR checkpoint. Produces detection precision/recall/F1, AP50, mAP50:95, mean IoU, inference time, end-to-end exact plate match, character accuracy, per-image results, and CSV/JSON reports.

**Important:** The notebook attempts filename-based vehicle grouping, but this is only a heuristic. Read the printed note before claiming the sample contains 100 distinct vehicles. Run this notebook and use its actual output values in your report; do not invent results.


In [1]:
# CELL 1 — Setup, Drive, paths
%pip -q install ultralytics opencv-python-headless pandas numpy matplotlib tqdm

import os, re, glob, json, time, random, warnings, sys, subprocess
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, cv2, torch, matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import fasterrcnn_resnet50_fpn, ssd300_vgg16
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from ultralytics import YOLO
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/UFPR_ALPR_Project"
DATASET = f"{PROJECT}/yolo_dataset"
SOURCE_DIR = "/content/drive/MyDrive/UFPR-ALPR dataset"
YOLO_PATH = f"{PROJECT}/checkpoints/yolov8_plate_detect/weights/best.pt"
FRCNN_PATH = f"{PROJECT}/faster_rcnn/checkpoints/fasterrcnn_plate_best.pth"
SSD_PATH = f"{PROJECT}/ssd/checkpoints/ssd300_plate_best.pth"
PARSEQ_PATH = f"{PROJECT}/parseq_ocr/parseq_ufpr_best.pt"
TEST_IMAGE_DIR = next((p for p in [f"{DATASET}/images/test", f"{DATASET}/images/testing"] if os.path.isdir(p)), None)
TEST_LABEL_DIR = next((p for p in [f"{DATASET}/labels/test", f"{DATASET}/labels/testing"] if os.path.isdir(p)), None)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_IMAGES, SEED, CONF, IOU_THR = 100, 2026, 0.25, 0.50
OUT_DIR = f"{PROJECT}/evaluation_100_images"
os.makedirs(OUT_DIR, exist_ok=True)
print("Device:", DEVICE)
print("Test image split:", TEST_IMAGE_DIR)
print("Test labels split:", TEST_LABEL_DIR)
print("Test split present:", bool(TEST_IMAGE_DIR and TEST_LABEL_DIR))
print("Reports will be saved to:", OUT_DIR)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Mounted at /content/drive
Device: cuda
Test image split: /content/drive/MyDrive/UFPR_ALPR_Project/yolo_dataset/images/test
Test labels split: /content/drive/MyDrive/UFPR_ALPR_Project/yolo_dataset/labels/test
Test split present: True
Reports will be saved to: /content/drive/MyDrive/UFPR_ALPR_Project/evaluation_100_images


In [3]:

%pip install -q pytorch-lightning timm einops omegaconf hydra-core lmdb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.9 MB/s eta 0:00:00


In [6]:

# CELL 2 — Load all three detectors and fine-tuned PARSeq

import os
import sys
import subprocess
import torch

# Install PARSeq dependencies if missing
try:
    import pytorch_lightning
    import timm
    import einops
    import omegaconf
    import hydra
except ImportError:
    %pip install -q pytorch-lightning timm einops omegaconf hydra-core lmdb
    import pytorch_lightning
    import timm
    import einops
    import omegaconf
    import hydra

# Check required paths
required = {
    "YOLOv8": YOLO_PATH,
    "Faster R-CNN": FRCNN_PATH,
    "SSD300": SSD_PATH,
    "PARSeq OCR": PARSEQ_PATH,
    "test images": TEST_IMAGE_DIR,
    "test labels": TEST_LABEL_DIR,
}

missing = [
    f"{name}: {path}"
    for name, path in required.items()
    if not path or not os.path.exists(path)
]
if missing:
    raise FileNotFoundError(
        "Missing required paths:\n" + "\n".join(missing)
    )

# Load YOLOv8
yolo_model = YOLO(YOLO_PATH)

# Load Faster R-CNN
frcnn_model = fasterrcnn_resnet50_fpn(
    weights=None,
    weights_backbone=None
)
frcnn_model.roi_heads.box_predictor = FastRCNNPredictor(
    frcnn_model.roi_heads.box_predictor.cls_score.in_features,
    2
)

fc = torch.load(
    FRCNN_PATH, map_location="cpu", weights_only=False
)
frcnn_state = fc.get(
    "model_state",
    fc.get("model_state_dict", fc)
)
frcnn_model.load_state_dict(frcnn_state)
frcnn_model = frcnn_model.to(DEVICE).eval()

# Load SSD300
ssd_model = ssd300_vgg16(
    weights=None,
    weights_backbone=None,
    num_classes=2
)

sc = torch.load(
    SSD_PATH, map_location="cpu", weights_only=False
)
ssd_state = sc.get(
    "model_state",
    sc.get("model_state_dict", sc)
)
ssd_model.load_state_dict(ssd_state)
ssd_model = ssd_model.to(DEVICE).eval()

# Get the complete PARSeq repository
PARSEQ_REPO = "/content/parseq_repo"

if not os.path.isfile(os.path.join(PARSEQ_REPO, "hubconf.py")):
    if os.path.exists(PARSEQ_REPO):
        import shutil
        shutil.rmtree(PARSEQ_REPO)

    subprocess.run(
        [
            "git", "clone",
            "https://github.com/baudm/parseq.git",
            PARSEQ_REPO
        ],
        check=True
    )

# Verify repository configuration
for rel in [
    "configs/main.yaml",
    "configs/experiment/parseq.yaml",
    "configs/model/parseq.yaml",
]:
    if not os.path.isfile(os.path.join(PARSEQ_REPO, rel)):
        raise FileNotFoundError(
            f"PARSeq repository is missing: {rel}"
        )

# Ensure Python imports PARSeq from the cloned repository
if PARSEQ_REPO in sys.path:
    sys.path.remove(PARSEQ_REPO)
sys.path.insert(0, PARSEQ_REPO)

for name in list(sys.modules):
    if name == "strhub" or name.startswith("strhub."):
        del sys.modules[name]

from strhub.data.module import SceneTextDataModule

# Create the architecture without downloading pretrained weights
parseq_model = torch.hub.load(
    PARSEQ_REPO,
    "parseq",
    source="local",
    pretrained=False
)


# Load fine-tuned PARSeq checkpoint
pc = torch.load(
    PARSEQ_PATH,
    map_location="cpu",
    weights_only=True
)

# Extract model weights
if isinstance(pc, dict) and "model_state_dict" in pc:
    ps = pc["model_state_dict"]
elif isinstance(pc, dict) and "state_dict" in pc:
    ps = pc["state_dict"]
elif isinstance(pc, dict) and "model_state" in pc:
    ps = pc["model_state"]
else:
    ps = pc

# Remove training-framework prefixes, if present
clean_state = {}
for key, value in ps.items():
    if key.startswith("model."):
        key = key[len("model."):]
    if key.startswith("module."):
        key = key[len("module."):]
    clean_state[key] = value

# PARSeq's Lightning model expects weights under "model."
# Add that prefix only when the checkpoint contains bare PARSeq keys.
if any(k.startswith("encoder.") or k.startswith("decoder.")
       or k == "pos_queries" for k in clean_state):
    clean_state = {
        ("model." + k): v
        for k, v in clean_state.items()
    }

# Load weights and report any architecture mismatch
result = parseq_model.load_state_dict(clean_state, strict=False)

print("Missing keys:", len(result.missing_keys))
print("Unexpected keys:", len(result.unexpected_keys))

if result.missing_keys or result.unexpected_keys:
    print("Missing key examples:", result.missing_keys[:10])
    print("Unexpected key examples:", result.unexpected_keys[:10])
    raise RuntimeError(
        "PARSeq checkpoint does not match the model architecture. "
        "Check the checkpoint structure before continuing."
    )

parseq_model = parseq_model.to(DEVICE).eval()

parseq_transform = SceneTextDataModule.get_transform(
    parseq_model.hparams.img_size
)

print("Fine-tuned PARSeq OCR loaded successfully.")
print("Device:", DEVICE)


Missing keys: 0
Unexpected keys: 0
Fine-tuned PARSeq OCR loaded successfully.
Device: cuda


In [7]:
# CELL 3 — Check test split and select up to 100 images, one per inferred vehicle where possible
exts = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
all_images = sorted(p for p in glob.glob(os.path.join(TEST_IMAGE_DIR, "*"))
                    if os.path.isfile(p) and Path(p).suffix.lower() in exts)
if not all_images: raise FileNotFoundError(f"No test images found in {TEST_IMAGE_DIR}")

def vehicle_id(path):
    stem = Path(path).stem
    # Heuristic only: common vehicle_camera_frame filename patterns
    m = re.match(r"^(\d+)[_-](\d+)[_-](\d+)$", stem)
    if m: return "vehicle_" + m.group(1)
    m = re.match(r"^.*?vehicle[_-]?(\d+).*?$", stem, re.I)
    if m: return "vehicle_" + m.group(1)
    return None

rng = random.Random(SEED); rng.shuffle(all_images)
groups, unknown = defaultdict(list), []
for p in all_images:
    gid = vehicle_id(p)
    (groups[gid] if gid else unknown).append(p)
if groups:
    selected = [rng.choice(v) for v in groups.values()]
    rng.shuffle(selected)
    selected = selected[:N_IMAGES]
    if len(selected) < min(N_IMAGES, len(all_images)):
        selected += [p for p in all_images if p not in set(selected)][:N_IMAGES-len(selected)]
    group_note = f"Inferred {len(groups)} filename vehicle groups. One image per group selected where possible; remaining slots may include other images."
else:
    selected = all_images[:N_IMAGES]
    group_note = "Vehicle IDs could not be inferred reliably from filenames. These are random unique test images, NOT verified distinct vehicles."
selected = selected[:min(N_IMAGES, len(all_images))]
print("Images in test split:", len(all_images))
print("Images selected:", len(selected))
print(group_note)


Images in test split: 1800
Images selected: 100
Vehicle IDs could not be inferred reliably from filenames. These are random unique test images, NOT verified distinct vehicles.


In [8]:
# CELL 4 — Annotation, OCR and detector helpers
def load_gt(label_path, w, h):
    boxes=[]
    if os.path.isfile(label_path):
        for line in open(label_path, encoding="utf-8"):
            a=line.split()
            if len(a)<5: continue
            cls,xc,yc,bw,bh=map(float,a[:5])
            if int(cls)!=0: continue
            boxes.append([max(0,(xc-bw/2)*w),max(0,(yc-bh/2)*h),min(w,(xc+bw/2)*w),min(h,(yc+bh/2)*h)])
    return np.asarray(boxes,dtype=float).reshape(-1,4)

ann_index={}
for split in ("training","validation","testing","test"):
    folder=os.path.join(SOURCE_DIR,split)
    if os.path.isdir(folder):
        for p in glob.glob(os.path.join(folder,"**","*.txt"),recursive=True): ann_index[Path(p).stem]=p

def actual_text(path):
    stem=Path(path).stem
    for name in [stem, stem.split("__",1)[-1]]:
        p=ann_index.get(name)
        if p:
            for line in open(p,encoding="utf-8"):
                if line.strip().lower().startswith("plate:"): return line.split(":",1)[1].strip().upper()
    return None

def norm(s): return re.sub(r"[^A-Z0-9]","",str(s).upper())
def edit_distance(a,b):
    a,b=norm(a),norm(b); prev=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        cur=[i]
        for j,y in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))
        prev=cur
    return prev[-1]
def char_acc(a,b): return max(0,1-edit_distance(a,b)/max(1,len(norm(a))))

def iou(a,b):
    x1,y1=max(a[0],b[0]),max(a[1],b[1]); x2,y2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]); ab=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(aa+ab-inter) if aa+ab-inter>0 else 0

def infer_yolo(path):
    r=yolo_model.predict(path,conf=CONF,iou=0.5,verbose=False,device=0 if DEVICE.type=="cuda" else "cpu")[0]
    out=[]
    if r.boxes is not None:
        for b in r.boxes:
            if int(b.cls.item())==0: out.append({"box":b.xyxy[0].cpu().numpy().astype(float),"score":float(b.conf.item())})
    return out

def infer_tv(det,path):
    im=cv2.imread(path)
    x=to_tensor(Image.fromarray(cv2.cvtColor(im,cv2.COLOR_BGR2RGB))).to(DEVICE)
    with torch.inference_mode(): out=det([x])[0]
    return [{"box":b.cpu().numpy().astype(float),"score":float(s.item())}
            for b,s,c in zip(out["boxes"],out["scores"],out["labels"])
            if int(c.item())==1 and float(s.item())>=CONF]

def ocr(crop):
    x=parseq_transform(Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))).unsqueeze(0).to(DEVICE)
    with torch.inference_mode(): logits=parseq_model(x); dec=parseq_model.tokenizer.decode(logits.softmax(-1))
    d=dec[0]
    return str(d[0] if isinstance(d,(list,tuple)) else d).strip().upper()

def match_counts(gt,preds,thr=.5):
    used=set(); tp=0; matched_ious=[]
    for p in sorted(preds,key=lambda z:z["score"],reverse=True):
        candidates=[(iou(p["box"],g),j) for j,g in enumerate(gt) if j not in used]
        best,j=max(candidates,default=(0,None))
        if j is not None and best>=thr: used.add(j); tp+=1; matched_ious.append(best)
    return tp,len(preds)-tp,len(gt)-tp,matched_ious

print("Helpers ready. Original plate-text annotations indexed:",len(ann_index))


Helpers ready. Original plate-text annotations indexed: 4500


In [9]:
# CELL 5 — Evaluate same selected images with all three detectors + shared PARSeq
detectors={"YOLOv8":infer_yolo,"Faster R-CNN":lambda p:infer_tv(frcnn_model,p),"SSD300":lambda p:infer_tv(ssd_model,p)}
records=[]; det_rows={k:[] for k in detectors}; times={k:[] for k in detectors}; gt_map={}
for path in tqdm(selected,desc="Testing images"):
    im=cv2.imread(path)
    if im is None: continue
    h,w=im.shape[:2]; name=os.path.basename(path)
    gt=load_gt(os.path.join(TEST_LABEL_DIR,Path(path).stem+".txt"),w,h); gt_map[name]=gt.tolist()
    actual=actual_text(path)
    for dname,fn in detectors.items():
        t0=time.perf_counter(); preds=sorted(fn(path),key=lambda z:z["score"],reverse=True); elapsed=time.perf_counter()-t0
        times[dname].append(elapsed)
        tp,fp,fnn,matched=match_counts(gt,preds,IOU_THR)
        for p in preds: det_rows[dname].append({"image":name,"box":p["box"].tolist(),"score":p["score"]})
        predicted="NO PLATE DETECTED"
        if preds:
            x1,y1,x2,y2=np.round(preds[0]["box"]).astype(int)
            x1,x2=max(0,x1),min(w,x2); y1,y2=max(0,y1),min(h,y2)
            if x2>x1 and y2>y1: predicted=ocr(im[y1:y2,x1:x2])
        exact=(norm(actual)==norm(predicted)) if actual is not None and predicted!="NO PLATE DETECTED" else False
        ca=char_acc(actual,predicted) if actual is not None and predicted!="NO PLATE DETECTED" else (0.0 if actual is not None else np.nan)
        records.append({"image":name,"detector":dname,"gt_boxes":len(gt),"pred_boxes":len(preds),
                        "TP_IoU50":tp,"FP_IoU50":fp,"FN_IoU50":fnn,"actual_plate":actual,
                        "predicted_plate":predicted,"exact_match":exact if actual is not None else np.nan,
                        "character_accuracy":ca,"inference_seconds":elapsed,"inference_ms":elapsed*1000,
                        "mean_matched_iou":float(np.mean(matched)) if matched else np.nan})
per_image=pd.DataFrame(records)
per_image.to_csv(f"{OUT_DIR}/per_image_results.csv",index=False)
display(per_image.head(12))
print("Per-image results saved.")


Testing images:   0%|          | 0/100 [00:00<?, ?it/s]

,image,detector,gt_boxes,pred_boxes,TP_IoU50,FP_IoU50,FN_IoU50,actual_plate,predicted_plate,exact_match,character_accuracy,inference_seconds,inference_ms,mean_matched_iou
0,track0097__track0097[23].png,YOLOv8,1,1,1,0,0,AYP7941,AYP41,False,0.714286,2.823660,2823.660322,0.749643
1,track0097__track0097[23].png,Faster R-CNN,1,1,1,0,0,AYP7941,AYPA1941,False,0.714286,0.496015,496.015041,0.792721
2,track0097__track0097[23].png,SSD300,1,0,0,0,1,AYP7941,NO PLATE DETECTED,False,0.000000,0.164918,164.918282,NaN
3,track0149__track0149[21].png,YOLOv8,1,1,1,0,0,AXW8502,AXW8502,True,1.000000,0.054548,54.548256,0.956316
4,track0149__track0149[21].png,Faster R-CNN,1,5,1,4,0,AXW8502,AXW8502,True,1.000000,0.318613,318.612516,0.930224
5,track0149__track0149[21].png,SSD300,1,1,1,0,0,AXW8502,AXW8502,True,1.000000,0.102415,102.415414,0.843053
6,track0093__track0093[02].png,YOLOv8,1,1,1,0,0,AZZ6958,AZZ6958,True,1.000000,0.056605,56.605236,0.818598
7,track0093__track0093[02].png,Faster R-CNN,1,1,1,0,0,AZZ6958,AZZ6958,True,1.000000,0.341681,341.680604,0.906029
8,track0093__track0093[02].png,SSD300,1,1,1,0,0,AZZ6958,AZZ6958,True,1.000000,0.103007,103.007198,0.705439
9,track0128__track0128[05].png,YOLOv8,1,1,1,0,0,AWP0816,AMP0816,False,0.857143,0.062150,62.149820,0.930313


Per-image results saved.


In [10]:
# CELL 6 — Calculate detection AP and summary metrics
def ap_at(rows, gt_by_image, threshold):
    total_gt=sum(len(v) for v in gt_by_image.values())
    if total_gt==0: return np.nan
    ranked=sorted(rows,key=lambda r:r["score"],reverse=True)
    used={k:set() for k in gt_by_image}; tps=[]; fps=[]
    for r in ranked:
        gts=gt_by_image.get(r["image"],[]); best,bi=0,None
        for j,g in enumerate(gts):
            if j in used.setdefault(r["image"],set()): continue
            val=iou(r["box"],g)
            if val>best: best,bi=val,j
        if bi is not None and best>=threshold: used[r["image"]].add(bi); tps.append(1); fps.append(0)
        else: tps.append(0); fps.append(1)
    if not tps: return 0.0
    tp=np.cumsum(tps); fp=np.cumsum(fps); rec=tp/total_gt; prec=tp/np.maximum(tp+fp,1e-12)
    return float(np.mean([prec[rec>=r].max() if np.any(rec>=r) else 0 for r in np.linspace(0,1,101)]))

detection=[]
for name in detectors:
    sub=per_image[per_image.detector==name]
    tp,fp,fnn=int(sub.TP_IoU50.sum()),int(sub.FP_IoU50.sum()),int(sub.FN_IoU50.sum())
    p=tp/(tp+fp) if tp+fp else 0; r=tp/(tp+fnn) if tp+fnn else 0
    f1=2*p*r/(p+r) if p+r else 0
    aps=[ap_at(det_rows[name],gt_map,t) for t in np.arange(.5,.96,.05)]
    detection.append({"detector":name,"TP":tp,"FP":fp,"FN":fnn,"precision_IoU50":p,"recall_IoU50":r,"F1_IoU50":f1,
                      "AP50":ap_at(det_rows[name],gt_map,.5),"mAP50_95":float(np.nanmean(aps)),
                      "mean_matched_IoU":float(sub.mean_matched_iou.mean()),
                      "mean_inference_ms_per_image":float(np.mean(times[name])*1000),
                      "total_inference_seconds":float(np.sum(times[name])),"images_evaluated":len(times[name]),
                      "GT_boxes":sum(len(v) for v in gt_map.values())})
det_metrics=pd.DataFrame(detection)
det_metrics.to_csv(f"{OUT_DIR}/detection_metrics.csv",index=False)
display(det_metrics)


,detector,TP,FP,FN,precision_IoU50,recall_IoU50,F1_IoU50,AP50,mAP50_95,mean_matched_IoU,mean_inference_ms_per_image,total_inference_seconds,images_evaluated,GT_boxes
0,YOLOv8,97,4,3,0.960396,0.97,0.965174,0.969192,0.722616,0.873757,85.506187,8.550619,100,100
1,Faster R-CNN,100,31,0,0.763359,1.00,0.865801,0.999216,0.792621,0.890732,280.174774,28.017477,100,100
2,SSD300,57,10,43,0.850746,0.57,0.682635,0.551639,0.280211,0.751940,109.652362,10.965236,100,100


In [11]:
# CELL 7 — End-to-end ALPR metrics and export report tables
alpr=[]
for name in detectors:
    sub=per_image[per_image.detector==name]
    labeled=sub[sub.actual_plate.notna()]
    exact=labeled.exact_match.dropna().astype(bool)
    ca=labeled.character_accuracy.dropna()
    alpr.append({"detector":name,"images_evaluated":len(sub),"images_with_plate_text":len(labeled),
                 "exact_match_correct":int(exact.sum()),"end_to_end_exact_match_accuracy":float(exact.mean()) if len(exact) else np.nan,
                 "mean_character_accuracy":float(ca.mean()) if len(ca) else np.nan,
                 "no_detection_count":int((sub.predicted_plate=="NO PLATE DETECTED").sum())})
alpr_metrics=pd.DataFrame(alpr)
alpr_metrics.to_csv(f"{OUT_DIR}/end_to_end_alpr_metrics.csv",index=False)
comparison=det_metrics.merge(alpr_metrics,on="detector")
comparison.to_csv(f"{OUT_DIR}/three_model_comparison.csv",index=False)
summary={"dataset":"UFPR-ALPR","test_images_dir":TEST_IMAGE_DIR,"test_labels_dir":TEST_LABEL_DIR,
         "selected_images":len(selected),"seed":SEED,"confidence_threshold":CONF,"IoU_threshold":IOU_THR,
         "vehicle_grouping_note":group_note,"detection_metrics":detection,"end_to_end_metrics":alpr,
         "report_directory":OUT_DIR}
with open(f"{OUT_DIR}/evaluation_summary.json","w",encoding="utf-8") as f: json.dump(summary,f,indent=2,allow_nan=True)
print("END-TO-END ALPR METRICS"); display(alpr_metrics)
print("COMBINED MODEL COMPARISON"); display(comparison)
print("Saved reports:")
for fn in ["per_image_results.csv","detection_metrics.csv","end_to_end_alpr_metrics.csv","three_model_comparison.csv","evaluation_summary.json"]:
    print(os.path.join(OUT_DIR,fn))


END-TO-END ALPR METRICS


,detector,images_evaluated,images_with_plate_text,exact_match_correct,end_to_end_exact_match_accuracy,mean_character_accuracy,no_detection_count
0,YOLOv8,100,100,73,0.73,0.894286,2
1,Faster R-CNN,100,100,75,0.75,0.928571,0
2,SSD300,100,100,46,0.46,0.577143,34


COMBINED MODEL COMPARISON


,detector,TP,FP,FN,precision_IoU50,recall_IoU50,F1_IoU50,AP50,mAP50_95,mean_matched_IoU,mean_inference_ms_per_image,total_inference_seconds,images_evaluated_x,GT_boxes,images_evaluated_y,images_with_plate_text,exact_match_correct,end_to_end_exact_match_accuracy,mean_character_accuracy,no_detection_count
0,YOLOv8,97,4,3,0.960396,0.97,0.965174,0.969192,0.722616,0.873757,85.506187,8.550619,100,100,100,100,73,0.73,0.894286,2
1,Faster R-CNN,100,31,0,0.763359,1.00,0.865801,0.999216,0.792621,0.890732,280.174774,28.017477,100,100,100,100,75,0.75,0.928571,0
2,SSD300,57,10,43,0.850746,0.57,0.682635,0.551639,0.280211,0.751940,109.652362,10.965236,100,100,100,100,46,0.46,0.577143,34


Saved reports:
/content/drive/MyDrive/UFPR_ALPR_Project/evaluation_100_images/per_image_results.csv
/content/drive/MyDrive/UFPR_ALPR_Project/evaluation_100_images/detection_metrics.csv
/content/drive/MyDrive/UFPR_ALPR_Project/evaluation_100_images/end_to_end_alpr_metrics.csv
/content/drive/MyDrive/UFPR_ALPR_Project/evaluation_100_images/three_model_comparison.csv
/content/drive/MyDrive/UFPR_ALPR_Project/evaluation_100_images/evaluation_summary.json


## Reporting guidance

Use `three_model_comparison.csv` for the main results table and `per_image_results.csv` for error analysis. State the number of test images and the exact checkpoint paths in your report.

**Interpretation cautions**
- This notebook does not claim the selected files are 100 distinct vehicles unless filename grouping actually supports that. Verify the printed note.
- AP/mAP is calculated here for one plate class using a 101-point interpolated AP implementation; for formal benchmarking, cross-check against a standard evaluator.
- OCR text accuracy is calculated only where original plate text annotations match image filenames. Images without a matched text annotation are excluded from OCR denominator; missed detections on labeled images count as incorrect.
- Detection metrics use YOLO labels in the test split. Confirm labels are present and correspond to the images.
- Do not use the test split to tune models after reviewing these results.
